# NIPS CITE-seq — Preprocessing Pipeline

Paired CITE-seq (NeurIPS 2021, 12 batches). Produces two preprocessed objects:

| Output | Modality | Pipeline |
|---|---|---|
| `adata_rna_proc` | GEX (RNA) | counts → HVG (Seurat v3) → normalize_total → log1p → scale (clip ±10) |
| `adata_adt_proc` | ADT (protein) | counts → **CLR** (no clipping) |

HVG selection and scaling are exposed as **functions** with `global` / `by_batch`
options; the actual run below uses the **conventional defaults**
(Global HVG, global scaling, CLR axis=0 for ADT).

In [ ]:
# !pip install scanpy scikit-misc anndata   # scikit-misc is required for flavor='seurat_v3'

import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy import sparse

## 1. Load data

The object is a *paired* CITE matrix: `var['feature_types']` is `GEX` or `ADT`, and `layers['counts']` holds raw integer counts for **all** features.

In [ ]:
# --- Colab ---
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/multi-omics/fullbatch.h5ad"
adata = sc.read_h5ad(path)
print(adata)

## 2. Clean & sanity-check

Make var names unique, confirm a raw `counts` layer exists, and verify counts are non-negative integers. We never touch `adata.X` directly — every pipeline starts from `layers['counts']`.

In [ ]:
adata.var_names_make_unique()

assert "counts" in adata.layers, "Need a raw counts layer."
assert "feature_types" in adata.var, "var['feature_types'] (GEX/ADT) required."

print(adata.var["feature_types"].value_counts())

C = adata.layers["counts"]
Cs = C[:2000, :500].toarray() if sparse.issparse(C) else np.asarray(C[:2000, :500])
print("counts  min/max:", Cs.min(), Cs.max(),
      "| integer:", np.allclose(Cs, np.round(Cs)),
      "| any negative:", bool(np.any(Cs < 0)))

print(f"{adata.obs['cell_type'].nunique()} cell types, "
      f"{adata.obs['batch'].nunique()} batches")

### (Optional) Major cell-type grouping

Collapses fine labels into 7 major lineages — handy for stratified / worst-case metrics downstream. Unmapped labels fall into `Other`.

In [ ]:
cell_library = {
    "monocyte":   ["CD14+ Mono", "CD16+ Mono"],
    "T cell":     ["CD4+ T naive", "CD4+ T activated", "CD4+ T activated integrinB7+",
                   "T reg", "CD8+ T naive", "CD8+ T naive CD127+ CD26- CD101-",
                   "CD8+ T CD49f+", "CD8+ T TIGIT+ CD45RO+", "CD8+ T CD57+ CD45RA+",
                   "CD8+ T CD69+ CD45RO+", "CD8+ T TIGIT+ CD45RA+", "CD8+ T CD69+ CD45RA+",
                   "CD8+ T CD57+ CD45RO+", "CD4+ T CD314+ CD45RA+", "dnT", "MAIT",
                   "gdT CD158b+", "gdT TCRVD2+"],
    "NK/ILC":     ["NK", "NK CD158e1+", "ILC", "ILC1"],
    "B cell":     ["Naive CD20+ B IGKC+", "Naive CD20+ B IGKC-", "B1 B IGKC-",
                   "B1 B IGKC+", "Transitional B", "Plasmablast IGKC+",
                   "Plasmablast IGKC-", "Plasma cell IGKC+", "Plasma cell IGKC-"],
    "DC":         ["cDC1", "cDC2", "pDC"],
    "Progenitor": ["HSC", "Lymph prog", "G/M prog", "MK/E prog", "T prog cycling"],
    "Erythroid":  ["Erythroblast", "Proerythroblast", "Normoblast", "Reticulocyte"],
}
fine2major = {f: m for m, fs in cell_library.items() for f in fs}
adata.obs["cell_type_major"] = adata.obs["cell_type"].map(fine2major).fillna("Other")
print(adata.obs["cell_type_major"].value_counts())



### In the same time，you may also try a more rough classification as following：
cell_library2 = {
    "monocyte": ["CD14+ Mono", "CD16+ Mono"], 
    "T cell": ["CD4+ T naive",
"CD4+ T activated",
"CD4+ T activated integrinB7+",
"T reg",
"CD8+ T naive",
"CD8+ T naive CD127+ CD26- CD101-",
"CD8+ T CD49f+",
"CD8+ T TIGIT+ CD45RO+",
"CD8+ T CD57+ CD45RA+",
"CD8+ T CD69+ CD45RO+",
"CD8+ T TIGIT+ CD45RA+",
"CD8+ T CD69+ CD45RA+",
"CD8+ T CD57+ CD45RO+",
"CD4+ T CD314+ CD45RA+",
"dnT"],
    "NK":["NK CD158e1+",
"NK"] ,
    "B cell":["Naive CD20+ B IGKC+",
"Naive CD20+ B IGKC-",
"B1 B IGKC-",
"B1 B IGKC+",
"Transitional B",
"Plasmablast IGKC+",
"Plasmablast IGKC-",
"Plasma cell IGKC+",
"Plasma cell IGKC-"]
}

## 3. Reusable functions

Both options, `global` vs `by_batch` are provided, but the run in §4 uses conventional defaults.

In [ ]:
def select_hvg(adata_gex, n_hvg=2000, mode="global",
               batch_key="batch", layer="counts"):
    """Highly variable genes on GEX *counts* using Seurat v3.
    mode='batch_aware' ranks HVGs within each batch then merges (scIB default);
    mode='global' ignores batch.
    """
    tmp = adata_gex.copy()
    kwargs = dict(n_top_genes=n_hvg, layer=layer, flavor="seurat_v3")
    if mode == "batch_aware":
        kwargs["batch_key"] = batch_key
    elif mode != "global":
        raise ValueError("mode must be 'batch_aware' or 'global'")
    sc.pp.highly_variable_genes(tmp, **kwargs)
    return tmp.var_names[tmp.var["highly_variable"]].tolist()


def _scale_by_batch(X, batch_labels, max_value=None):
    X = np.asarray(X, dtype=np.float32)
    out = np.empty_like(X)
    bl = pd.Series(batch_labels).astype(str).values
    for b in np.unique(bl):
        idx = np.where(bl == b)[0]
        sub = AnnData(X[idx].copy())
        sc.pp.scale(sub, max_value=max_value)
        out[idx] = sub.X
    return out


def scale_expression(adata, mode="global", batch_key="batch", max_value=None):
    """Z-score each gene. mode='global' (across all cells) or
    'by_batch' (z-scored independently within each batch).
    max_value clips after scaling; pass None to disable clipping.
    """
    X = adata.X.toarray() if sparse.issparse(adata.X) else np.asarray(adata.X)
    if mode == "global":
        sub = AnnData(X.copy())
        sc.pp.scale(sub, max_value=max_value)
        return np.asarray(sub.X, dtype=np.float32)
    if mode == "by_batch":
        return _scale_by_batch(X, adata.obs[batch_key].values, max_value=max_value)
    raise ValueError("mode must be 'global' or 'by_batch'")


def clr_transform(adata_adt, axis=0, inplace=False):
    """Centered log-ratio for ADT (matches muon.prot.pp.clr).
    axis=0: per-protein across cells (muon default / Seurat margin=2).
    axis=1: per-cell across proteins (canonical compositional CLR).
    Formula: log1p( x / exp(mean_axis(log1p(x))) ).
    """
    ad = adata_adt if inplace else adata_adt.copy()
    X = ad.X.toarray() if sparse.issparse(ad.X) else np.asarray(ad.X, dtype=np.float64)
    denom = np.exp(np.log1p(X).sum(axis=axis, keepdims=True) / X.shape[axis])
    ad.X = np.log1p(X / denom).astype(np.float32)
    return None if inplace else ad

## 4. GEX (RNA) preprocessing

Conventional path: subset GEX → HVG on counts → `normalize_total(1e4)` → `log1p` →
`scale` with clip ±10. The unscaled log-normalized matrix is kept in
`layers['lognorm']` so you can recover it without re-running.

In [ ]:
def preprocess_gex(adata, n_hvg=2000, hvg_mode="global", scale_mode="global",
                   batch_key="batch", target_sum=1e4, max_value= None):
    gex = adata[:, adata.var["feature_types"].astype(str) == "GEX"].copy()
    hvg = select_hvg(gex, n_hvg=n_hvg, mode=hvg_mode, batch_key=batch_key)
    gex = gex[:, hvg].copy()

    gex.X = gex.layers["counts"].copy()
    sc.pp.normalize_total(gex, target_sum=target_sum)
    sc.pp.log1p(gex)
    gex.layers["lognorm"] = gex.X.copy()              # unscaled log-norm
    gex.X = scale_expression(gex, mode=scale_mode,
                             batch_key=batch_key, max_value=max_value)

    gex.uns["preprocessing"] = dict(modality="GEX", n_hvg=n_hvg, hvg_mode=hvg_mode,
                                    scale_mode=scale_mode, target_sum=target_sum,
                                    clip=max_value)
    return gex, hvg


#### implement preprocessing on RNA data

adata_rna_proc, hvg_genes = preprocess_gex(
    adata,
    n_hvg=2000,
    hvg_mode="global",   # conventional; 'batch_aware' also available
    scale_mode="global",      # conventional; 'by_batch' also available
    batch_key="batch",
    max_value=None,
)
print(adata_rna_proc)
print("X mean ~0:", round(float(adata_rna_proc.X.mean()), 3),
      "| max:", round(float(adata_rna_proc.X.max()), 2))

## 5. ADT (protein) preprocessing — CLR

Protein counts are compositional, so we use **CLR**, not library-size log-norm.
We start from raw `layers['counts']`, apply CLR (`axis=0`, per-protein across cells),
and keep the result in `layers['clr']`. No clipping (CLR already compresses the range).
Optional post-CLR z-scoring is off by default.

In [ ]:
def preprocess_adt(adata, clr_axis=0, scale_adt=False,
                   scale_mode="global", batch_key="batch"):
    adt = adata[:, adata.var["feature_types"].astype(str) == "ADT"].copy()

    adt.X = adt.layers["counts"].copy()              # CLR on raw protein counts
    adt = clr_transform(adt, axis=clr_axis)
    adt.layers["clr"] = adt.X.copy()

    if scale_adt:                                    # optional, no clipping
        adt.X = scale_expression(adt, mode=scale_mode,
                                 batch_key=batch_key, max_value=None)

    adt.uns["preprocessing"] = dict(modality="ADT", norm="CLR",
                                    clr_axis=clr_axis, scaled=scale_adt)
    return adt


adata_adt_proc = preprocess_adt(adata, clr_axis=0, scale_adt=False)
print(adata_adt_proc)
print("proteins:", adata_adt_proc.n_vars,
      "| CLR range:", round(float(adata_adt_proc.X.min()), 2),
      "to", round(float(adata_adt_proc.X.max()), 2))

# paired objects share the same cells
assert (adata_rna_proc.obs_names == adata_adt_proc.obs_names).all()

## 6. (Optional) Sanity UMAP on RNA

In [ ]:
### try a simple visualization to see the low-rank embedding of RNA modalities across celltype/batch
sc.tl.pca(adata_rna_proc, n_comps=30, svd_solver="arpack")
sc.pp.neighbors(adata_rna_proc, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_rna_proc, random_state=0)
sc.pl.umap(adata_rna_proc, color=["batch", "cell_type"], wspace=0.4)

## 7. Save

In [ ]:
out_dir = "."   # e.g. "/content/drive/MyDrive/multi-omics/processed"
adata_rna_proc.write(f"{out_dir}/cite_nips_RNA_proc.h5ad")
adata_adt_proc.write(f"{out_dir}/cite_nips_ADT_proc.h5ad")
pd.DataFrame({"hvg_gene": hvg_genes}).to_csv(f"{out_dir}/cite_nips_hvg_genes.csv", index=False)
print("saved RNA, ADT, and HVG list.")